<div style="background: linear-gradient(135deg, #1e4d3a 0%, #3d8b6a 100%); color: white; padding: 28px 32px; border-radius: 10px;">
<h1 style="margin: 0; color: white;">Hyperparameter Optimization</h1>
<p style="margin: 8px 0 0 0; font-size: 1.1em; opacity: 0.9;">Grid search vs random search vs Bayesian optimization with Optuna</p>
<p style="margin: 14px 0 0 0; font-size: 0.9em; opacity: 0.75;">Carles Trullàs</p>
</div>

Choosing hyperparameters is one of the most expensive parts of building a model: every candidate configuration means training and cross-validating the model again. This notebook compares three strategies for exploring the **same search space** on the **same model**, and measures how good the best configuration is and how long it takes to find it.

| Strategy | How it picks the next configuration | Uses past results? |
|---|---|---|
| **Grid search** | Tries every combination in the grid, in order | No |
| **Random search** | Samples combinations at random | No |
| **Bayesian optimization (TPE)** | Builds a probabilistic model of which regions score well and samples there | Yes |

### Contents

1. [How Bayesian optimization works](#1.-How-Bayesian-optimization-works)
2. [Setup](#2.-Setup)
3. [Data, model and search space](#3.-Data,-model-and-search-space)
4. [Grid search](#4.-Grid-search)
5. [Random search](#5.-Random-search)
6. [Bayesian optimization with Optuna](#6.-Bayesian-optimization-with-Optuna)
7. [Comparison](#7.-Comparison)
8. [Takeaways](#8.-Takeaways)

## 1. How Bayesian optimization works

Grid and random search are **uninformed**: the 50th configuration they try is chosen without looking at how the first 49 performed. Bayesian optimization treats the cross-validation score as an unknown, expensive function $f(\theta)$ of the hyperparameters $\theta$, and uses every evaluation so far to decide where to look next.

Optuna's default sampler, the **Tree-structured Parzen Estimator (TPE)**, works like this:

1. Split the trials evaluated so far into a **good** group (the top $\gamma$ fraction by score) and a **bad** group (the rest).
2. Fit a density to each group: $\ell(\theta)$ for the good configurations and $g(\theta)$ for the bad ones.
3. Draw candidate configurations from $\ell(\theta)$ and pick the one that maximises

$$
\frac{\ell(\theta)}{g(\theta)},
$$

that is, the configuration that looks most like past winners and least like past losers.

4. Evaluate it, add it to the history, and repeat.

The first few trials are random (there is nothing to learn from yet), and as evidence accumulates the search concentrates on the promising regions. The result is usually a good configuration in far fewer evaluations than a grid would need.

## 2. Setup

In [ ]:
# Uncomment if Optuna is not installed in your environment
# %pip install optuna

In [ ]:
from time import perf_counter

import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
from optuna.samplers import TPESampler
from sklearn.datasets import load_digits
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, StratifiedKFold, cross_val_score

optuna.logging.set_verbosity(optuna.logging.WARNING)

SEED = 42
SCORING = "f1_micro"
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})

COLORS = {"Grid search": "#8d99ae", "Random search": "#e07a3f", "Bayesian (Optuna)": "#2f7d5a"}

## 3. Data, model and search space

The **digits** dataset: 1,797 grayscale images of handwritten digits (8×8 pixels, flattened to 64 features), with 10 classes. The model is a random forest, scored with micro-averaged F1 (equal to accuracy for single-label multiclass problems) using 5-fold stratified cross-validation.

In [ ]:
digits = load_digits()
X, y = digits.data, digits.target

print(f"X: {X.shape}  |  classes: {np.unique(y).size}")

fig, axes = plt.subplots(1, 10, figsize=(12, 1.6))
for ax, img, label in zip(axes, digits.images, digits.target):
    ax.imshow(img, cmap="gray_r")
    ax.set_title(label)
    ax.axis("off")
plt.show()

### Search space

Six hyperparameters with a few values each. The full grid has $4 \times 2 \times 3 \times 5 \times 2 \times 3 = 720$ combinations, which with 5-fold cross-validation means **3,600 model fits** for grid search.

In [ ]:
search_space = {
    "n_estimators": [50, 100, 150, 200],
    "criterion": ["gini", "entropy"],
    "min_samples_split": [2, 3, 4],
    "min_samples_leaf": [1, 2, 3, 4, 5],
    "max_features": ["sqrt", "log2"],
    "max_depth": [5, 6, 7],
}

grid_size = int(np.prod([len(v) for v in search_space.values()]))
print(f"Grid size: {grid_size} combinations")

### Shared helpers

`summarize` turns the result of any of the three strategies into the same row of a comparison table. `best_so_far` gives, after each evaluation, the best score found up to that point, which is what the convergence plot uses.

In [ ]:
results, histories = [], {}


def summarize(name, scores, best_params, elapsed):
    scores = np.asarray(scores)
    histories[name] = np.maximum.accumulate(scores)  # best score found after each evaluation
    best_idx = int(np.argmax(scores))
    results.append({
        "Strategy": name,
        "Evaluations": len(scores),
        "Time (s)": round(elapsed, 1),
        "Best CV score": round(scores.max(), 4),
        "Found at evaluation": best_idx + 1,
        "Best hyperparameters": best_params,
    })
    print(f"{name}: best score {scores.max():.4f} at evaluation {best_idx + 1}/{len(scores)} "
          f"in {elapsed:.1f} s")
    print(f"  {best_params}")

## 4. Grid search

Exhaustive: all 720 combinations. It is guaranteed to find the best configuration **within the grid**, at the highest possible cost.

In [ ]:
grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=SEED),
    param_grid=search_space,
    scoring=SCORING,
    cv=CV,
    n_jobs=-1,
)

start = perf_counter()
grid_search.fit(X, y)
summarize("Grid search", grid_search.cv_results_["mean_test_score"],
          grid_search.best_params_, perf_counter() - start)

## 5. Random search

To make the comparison fair, random search and Bayesian optimization get the **same budget: 100 evaluations**, about 14% of the grid.

In [ ]:
N_TRIALS = 100

random_search = RandomizedSearchCV(
    estimator=RandomForestClassifier(random_state=SEED),
    param_distributions=search_space,
    n_iter=N_TRIALS,
    scoring=SCORING,
    cv=CV,
    n_jobs=-1,
    random_state=SEED,
)

start = perf_counter()
random_search.fit(X, y)
summarize("Random search", random_search.cv_results_["mean_test_score"],
          random_search.best_params_, perf_counter() - start)

## 6. Bayesian optimization with Optuna

In Optuna the search space is defined inside an **objective function**: each `trial.suggest_*` call asks the sampler for a value, and the function returns the score to maximise. The space below is identical to the grid.

In [ ]:
def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 200, step=50),
        "criterion": trial.suggest_categorical("criterion", ["gini", "entropy"]),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 4),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 5),
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2"]),
        "max_depth": trial.suggest_int("max_depth", 5, 7),
    }
    model = RandomForestClassifier(**params, random_state=SEED)
    return cross_val_score(model, X, y, scoring=SCORING, cv=CV, n_jobs=-1).mean()


study = optuna.create_study(direction="maximize", sampler=TPESampler(seed=SEED))

start = perf_counter()
study.optimize(objective, n_trials=N_TRIALS)
summarize("Bayesian (Optuna)", [t.value for t in study.trials],
          study.best_params, perf_counter() - start)

### Where did the sampler look?

The value chosen for each hyperparameter across the 100 trials, coloured by score. As TPE learns, the trials should cluster around the values that work best.

In [ ]:
trials = study.trials_dataframe(attrs=("number", "value", "params"))
trials.columns = [c.replace("params_", "") for c in trials.columns]

fig, axes = plt.subplots(2, 3, figsize=(14, 6.5), sharey=True)
for ax, param in zip(axes.flat, search_space):
    sc = ax.scatter(trials["number"], trials[param].astype(str), c=trials["value"],
                    cmap="viridis", s=28, alpha=0.85)
    ax.set(title=param, xlabel="Trial")
fig.colorbar(sc, ax=axes, label="CV score", shrink=0.8)
plt.show()

In [ ]:
importances = pd.Series(optuna.importance.get_param_importances(study)).sort_values()

ax = importances.plot.barh(figsize=(8, 3.5), color=COLORS["Bayesian (Optuna)"])
ax.set(xlabel="Relative importance", title="Hyperparameter importance (fANOVA)")
plt.tight_layout()
plt.show()

## 7. Comparison

In [ ]:
summary = pd.DataFrame(results).set_index("Strategy")
summary["Time vs grid"] = (summary["Time (s)"] / summary.loc["Grid search", "Time (s)"]).map("{:.0%}".format)

summary[["Evaluations", "Time (s)", "Time vs grid", "Best CV score", "Found at evaluation"]]

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5), gridspec_kw={"width_ratios": [2, 1]})

# Convergence: best score found after each evaluation
for name, hist in histories.items():
    ax1.plot(np.arange(1, len(hist) + 1), hist, lw=2, color=COLORS[name], label=name)
ax1.set(xscale="log", xlabel="Number of evaluations (log scale)", ylabel="Best CV score so far",
        title="Convergence")
ax1.legend(frameon=False, loc="lower right")

# Total time
ax2.barh(summary.index, summary["Time (s)"], color=[COLORS[n] for n in summary.index])
for i, t in enumerate(summary["Time (s)"]):
    ax2.text(t, i, f" {t:.0f} s", va="center")
ax2.invert_yaxis()
ax2.set(xlabel="Seconds", title="Total search time")
ax2.grid(axis="y", visible=False)

plt.tight_layout()
plt.show()

Grid search walks through the grid in a fixed order, so its convergence curve depends on where the best region happens to sit in that order. Random search improves quickly at first and then flattens. Bayesian optimization typically keeps improving after the random warm-up phase, because each new trial is aimed at the regions that have scored well so far.

## 8. Takeaways

- **Grid search is exhaustive but scales badly.** Its cost is the product of the number of values of every hyperparameter, so adding one more hyperparameter or one more value multiplies the runtime.
- **Random search is a strong baseline.** With a fraction of the budget it usually gets close to the best grid score, because only a few hyperparameters tend to matter and random sampling covers each of them well.
- **Bayesian optimization uses its budget more efficiently.** By learning from past trials, TPE concentrates the evaluations where good configurations are, and tends to match or beat random search with the same number of trials.
- **The gap grows with the cost of each evaluation.** On a small dataset like digits every strategy finishes in minutes; on a large model where a single fit takes an hour, needing 100 evaluations instead of 720 is the difference between a day and a month.
- **Optuna also handles continuous and conditional spaces.** Unlike a grid, it can search learning rates on a log scale or hyperparameters that only exist for some model choices, and it supports pruning unpromising trials early.